google-genai the official gemini sdk

In [22]:
%pip install python-dotenv google-genai


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Sentence-transformer : bi-encoder and corss-encoder models 
faiss-cpu : the vector store
pypdf :reads the text out of the pdf


In [23]:
%pip install -q sentence-transformers faiss-cpu pypdf


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
%pip install pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Cell 2.2


In [25]:
import re, time, textwrap
import numpy as np
import pandas as pd
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

# ---- settings: every experiment knob lives here ----
PDF_PATH = "document.pdf"          # GDPR, Regulation (EU) 2016/679, from EUR-Lex

CHUNK_WORDS = 180                  # words per chunk
OVERLAP_WORDS = 40                 # words shared by neighbouring chunks

BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # stage 1: bi-encoder, 384-dim vectors
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"   # stage 2: cross-encoder re-ranker (the brief's example)
LLM_MODEL = "gemini-2.5-flash"     # free tier: 5 requests/minute, 20 requests/day

TOP_K_RETRIEVE = 10                # candidates returned by the bi-encoder
TOP_N_CONTEXT = 3                  # chunks given to the LLM in both pipelines

pd.set_option("display.max_colwidth", 120)

We chunk for two reasons. An LLM prompt has limited room, and one embedding can only describe a short passage well: a 3-page chunk would blur ten rules into one vague vector. Chunks that are too small lose context ("this rule" with no subject). About 150 to 250 words is a sensible middle. We use a sliding window: each chunk starts 140 words after the previous one, so neighbours share 40 words and a sentence cut at one boundary appears whole in the next chunk. We keep the page number on every chunk, so you can cite where an answer came from.

In [26]:
def load_pdf_pages(path):
    """
    Return a lsit of {'pages':n, 'text': ... } for every page that has
    text"""

    reader = PdfReader(path)

    pages = []

    for i ,page in enumerate(reader.pages):
        text = page.extract_text() or "" 
        text = re.sub(r"\s+"," ", text).strip() # collaspe  newlines and repeated spaces

        if text:
            pages.append({"page":i+1, "text": text}) 
    return pages

pages = load_pdf_pages(PDF_PATH)
print(f"{len(pages)} pages with text")

print(pages[0]["text"][:500]) #the first page: check if text is readable or not 

88 pages with text
I (Legislative acts) REGULATIONS REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 27 April 2016 on the protection of natural persons with regard to the processing of personal data and on the free movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 1


cell 3.2 : split pages into overlapping chunks 

In [27]:
def chunk_pages(pages, chunk_words = CHUNK_WORDS, overlap = OVERLAP_WORDS):
    """
    SLIDE A window of chunk_words over each page and moving (chunk_words- overlap ) words each step.
    """

    chunks = []
    step = chunk_words - overlap

    for p in pages:
        words = p["text"].split()
        for start in range(0, len(words), step):
            piece = words[start: start+chunk_words]
            chunks.append({
                "chunk_id":len(chunks), #position in the list = id used everywhere later
                "page":p["page"],
                "text":" ".join(piece),
            })
            if start + chunk_words  >= len(words):
                break
    
    return chunks


chunks = chunk_pages(pages)
lens = [len(c["text"].split()) for c in chunks]
print(f"{len(pages)} pages -> {len(chunks)} chunks")

print("words per chunk : min", min(lens), "max", max(lens), 
"avg", round(sum(lens) /len(lens)))


88 pages -> 419 chunks
words per chunk : min 45 max 180 avg 165


Cell 3.3 : two helpers we willues in every later chapter

In [28]:
def show_chunk(cid):
    """print one chunks in full , wrapped for reading."""
    print(f"  chunk {cid} (page {chunks[cid]['page']})") 
    
    print(textwrap.fill(chunks[cid]["text"], 110))


def find_chunk(keyword):
    """plain keyword search over chunks : how we localt eh gold chunk for a quesiton. 
    """

    hits  =  [c for c in chunks if keyword.lower() in c["text"].lower()]

    return pd.DataFrame([{"chunk_id": c["chunk_id"], "page":  c["page"], "preview": c["text"][:150]} for c in hits])


show_chunk(0)
find_chunk("except")


  chunk 0 (page 1)
I (Legislative acts) REGULATIONS REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 27
April 2016 on the protection of natural persons with regard to the processing of personal data and on the free
movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation) (Text with EEA
relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the
Functioning of the European Union, and in particular Article 16 thereof, Having regard to the proposal from
the European Commission, After transmission of the draft legislative act to the national parliaments, Having
regard to the opinion of the European Economic and Social Committee (1), Having regard to the opinion of the
Committee of the Regions (2), Acting in accordance with the ordinary legislative procedure (3), Whereas: (1)
The protection of natural persons in relation to the processing of personal data is a fundamental right.
Art

,chunk_id,page,preview
0,34,7,"the controller and the processor, the competent lead supervisory authority should remain the supervisory authority o..."
1,104,18,impact assessments and from prior consultation of the supervisory authority. (96) A consultation of the supervisory ...
2,200,36,"compliance with, paragraph 1 (‘accountability’). Article 6 Lawfulness of processing 1. Processing shall be lawful on..."
3,201,36,in the exercise of official authority vested in the controller; (f) processing is necessary for the purposes of the ...
4,208,38,2. The controller shall make reasonable efforts to verify in such cases that consent is given or authorised by the h...
5,209,38,"data subject has given explicit consent to the processing of those personal data for one or more specified purposes,..."
6,215,39,"identification of a data subject by the controller, the controller shall not be obliged to maintain, acquire or proc..."
7,237,44,subject opposes the erasure of the personal data and requests the restriction of their use instead; (c) the controll...
8,259,50,The Commission may lay down standard contractual clauses for the matters referred to in paragraph 3 and 4 of this Ar...
9,260,50,"and any person acting under the authority of the controller or of the processor, who has access to personal data, sh..."


This chapter embeds every chunk once, stores the vectors in FAISS, and writes the function that returns the top 10 chunks for any query.
Idea
all-MiniLM-L6-v2 maps any text to 384 numbers. We normalize every vector to length 1; after that, the inner product of two vectors equals their cosine similarity, which is why we can use FAISS's IndexFlatIP ("flat" = compare against every vector exactly, "IP" = inner product). For a few hundred or a few thousand chunks, exact search takes well under a millisecond. Approximate indexes such as HNSW only matter at millions of vectors.
Notice the split in cost: encoding all chunks happens once, before any question. At question time only the short query is encoded, and FAISS does the comparison.

In [29]:
# download the model on first run 
bi_encoder  = SentenceTransformer(BI_ENCODER_NAME)

chunk_texts = [c["text"] for c in chunks]

#one vector per chunk.  Normalize_embeddings = true will make inner product == cosine similarity . 

chunk_embeddings = bi_encoder.encode(
    chunk_texts, batch_size = 32, normalize_embeddings=True
).astype("float32")  #FAISS expects float32

index = faiss.IndexFlatIP(chunk_embeddings.shape[1]) # exact inner-product search

index.add(chunk_embeddings)
#for i of the index = chunk_id i 
print("vectors in index:" , index.ntotal, " | dimensions:", chunk_embeddings.shape[1])




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12883.23it/s]


vectors in index: 419  | dimensions: 384


cell 4.2 baseline retrieval fuction

In [30]:
def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """
    Pipeline 1 retrieval: embed the query , return the k most similar chunks with score and rank. """

    q_emb = bi_encoder.encode([query],
    normalize_embeddings=True).astype("float32")

    scores, ids = index.search(q_emb,k) # both have shape(1,k) , best first

    return [
        {"chunk_id" : int(cid),
        "bi_score": float(s), 
        "bi_rank":rank+1}
        for rank, (cid, s) in enumerate(zip(ids[0], scores[0])) if cid != -1  
        #Faiss pads with -1 if fewer than k chunks exists. 
    ]


for c in baseline_retrieve("What rights do I have over my personal data?"):
    print(c["bi_rank"], round(c["bi_score"], 3), " | page ",  chunks[c["chunk_id"]]["page"], " |" ,
    chunks[c["chunk_id"]]["text"][:80])

1 0.688  | page  43  | Article 15 Right of access by the data subject 1. The data subject shall have th
2 0.675  | page  12  | online services and online identifiers. A controller should not retain personal 
3 0.674  | page  12  | origin of the personal data cannot be provided to the data subject because vario
4 0.638  | page  12  | access to personal data which have been collected concerning him or her, and to 
5 0.635  | page  13  | his or her consent as a child and is not fully aware of the risks involved by th
6 0.625  | page  2  | (4) The processing of personal data should be designed to serve mankind. The rig
7 0.622  | page  12  | Where possible, the controller should be able to provide remote access to a secu
8 0.618  | page  13  | systems which are technically compatible. Where, in a certain set of personal da
9 0.609  | page  36  | (e) kept in a form which permits identification of data subjects for no longer t
10 0.604  | page  8  | in the public interest to do so, including

Replace the test question with one that fits you pdf 

Chapter 5: Cross-encoder re-ranking (Pipeline 2, steps a to c)
The re-ranker takes the bi-encoder's 10 candidates, scores each (query, chunk) pair jointly, and sorts them again.
Idea
ms-marco-MiniLM-L6-v2 was trained on MS MARCO, millions of real search queries labelled with the passage that answered them. Its input is one sequence, [CLS] query [SEP] chunk [SEP]; all six transformer layers let query words and chunk words attend to each other, and a final layer outputs one number. Three things to know:
• The score is usually a raw logit rather than a probability, so it can be negative, like -7.2 or 5.4 (some library versions squash it to 0 to 1 instead). Only the order matters.
• Input is capped at 512 tokens. Our 180-word chunks plus a question fit comfortably; very long chunks would be silently cut.
• Cost grows with the number of pairs: 10 candidates means 10 forward passes per question, and none can be cached, because each depends on the question.

Cell 5.1 load the cross-encoder and define re-ranking

In [31]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)   # about 90 MB download on first run

def rerank(query, candidates):
    """Pipeline 2: score each (query, chunk) pair jointly, then sort by that score, best first."""
    pairs = [(query, chunks[c["chunk_id"]]["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)             # one relevance logit per pair
    reranked = [dict(c, ce_score=float(s)) for c, s in zip(candidates, scores)]  # keep bi_rank, add ce_score
    reranked.sort(key=lambda c: c["ce_score"], reverse=True)
    for rank, c in enumerate(reranked):
        c["ce_rank"] = rank + 1
    return reranked

# Same test question as Chapter 4: compare the two orders
q = "What rights do I have over my personal data?"
for c in rerank(q, baseline_retrieve(q)):
    print(f"bi #{c['bi_rank']:>2} -> ce #{c['ce_rank']:>2} | ce_score {c['ce_score']:6.2f} |",
          chunks[c["chunk_id"]]["text"][:70])

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3889.51it/s]


bi # 4 -> ce # 1 | ce_score   5.01 | access to personal data which have been collected concerning him or he
bi # 6 -> ce # 2 | ce_score   4.42 | (4) The processing of personal data should be designed to serve mankin
bi # 8 -> ce # 3 | ce_score   4.28 | systems which are technically compatible. Where, in a certain set of p
bi # 2 -> ce # 4 | ce_score   3.96 | online services and online identifiers. A controller should not retain
bi # 1 -> ce # 5 | ce_score   3.93 | Article 15 Right of access by the data subject 1. The data subject sha
bi # 5 -> ce # 6 | ce_score   3.59 | his or her consent as a child and is not fully aware of the risks invo
bi # 3 -> ce # 7 | ce_score   2.62 | origin of the personal data cannot be provided to the data subject bec
bi #10 -> ce # 8 | ce_score   2.04 | in the public interest to do so, including for health purposes such as
bi # 7 -> ce # 9 | ce_score   1.04 | Where possible, the controller should be able to provide remote access
bi # 9 -> ce #10 | ce_score 

Chapter 6: LLM answer generation with an identical prompt
Both answers come from one function and one prompt template; the only thing that changes between Answer 1 and Answer 2 is which three chunks fill the context.
Idea
This is a controlled experiment. If the prompt, model or temperature differed between the pipelines, you could not say that re-ranking caused a better answer. So:
• One PROMPT_TEMPLATE, used for both answers.
• Temperature 0, so the model is as deterministic as possible and differences come from the context, not from randomness.
• The prompt tells the model to answer only from the context and to say "Not found" otherwise. This makes failures visible: a missing chunk shows up as "Not found" or as a confident answer that leaves out a condition.
• Each chunk is labelled with its id and page, and the model is asked to cite them, so you can trace which chunk produced which claim.

In [32]:
import os
from google import genai
from google.genai import types

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")



client = genai.Client(api_key=api_key) #key comes from .env

#model names change over time :lsit what your key can use and pick a 'flash' one from LLM_MODEL

for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


If gemini-2.5-flash is not in the list, set LLM_MODEL in Cell 2.2 to one that is, without the models/ prefix, and re-run Cell 2.2.

Cell 6.2: the prompt and the answer function

In [33]:
PROMPT_TEMPLATE = """You are a careful assistant answering questions about a single document.
Use ONLY the context passages below. Do not use outside knowledge.
If the passages do not contain the answer, reply exactly: "Not found in the provided context."
Mention any exceptions, conditions or penalties that the passages state.
Cite the chunk ids you used, e.g. [chunk 12].

Context:
{context}

Question: {question}
Answer:"""

def build_context(chunk_ids):
    """Join the chosen chunks, each labelled with its id and page, in the order given."""
    return "\n\n".join(f"[chunk {cid} | page {chunks[cid]['page']}]\n{chunks[cid]['text']}"
                       for cid in chunk_ids)

MIN_SECONDS_BETWEEN_CALLS = 13     # free tier allows 5 requests per minute -> at least 12 s apart
_last_call_time = 0.0

def generate_answer(question, chunk_ids, retries=6):
    """Fill the SAME template with the given chunks and ask the LLM.
    Returns (answer_text, llm_ms). llm_ms times only the successful call,
    so waiting for rate limits does not inflate the latency numbers."""
    global _last_call_time
    prompt = PROMPT_TEMPLATE.format(context=build_context(chunk_ids), question=question)
    for attempt in range(1, retries + 1):
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.perf_counter() - _last_call_time)
        if wait > 0:
            time.sleep(wait)                          # stay under 5 requests/minute
        t0 = time.perf_counter()
        _last_call_time = t0
        try:
            resp = client.models.generate_content(
                model=LLM_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0),
            )
            return resp.text, (time.perf_counter() - t0) * 1000
        except Exception as e:
            msg = str(e)
            if "PerDay" in msg:                       # daily quota used up: waiting a few seconds won't help
                raise RuntimeError("Daily Gemini quota is used up. Try again after it resets (about 5:30 am IST).") from e
            if attempt == retries:
                raise
            delay = min(30 * attempt, 90)             # 30, 60, 90 s ... for 429 per-minute and 503 'high demand'
            print(f"LLM call failed ({msg[:60]}...); retry {attempt}/{retries - 1} in {delay} s")
            time.sleep(delay)

# smoke test: chunks 266-267 are Article 33, so the answer should mention 72 hours
answer, ms = generate_answer("When must a personal data breach be reported?", [266, 267])
print(answer)
print(f"({ms:.0f} ms)")

A personal data breach must be reported by the controller to the supervisory authority without undue delay and, where feasible, not later than 72 hours after becoming aware of it, unless the breach is unlikely to result in a risk to the rights and freedoms of natural persons. If the notification is not made within 72 hours, it must be accompanied by reasons for the delay [chunk 266, chunk 267].

The processor must notify the controller without undue delay after becoming aware of a personal data breach [chunk 267].
(2534 ms)


Chapter 7: Running both pipelines and reading rank shifts
run_pipelines runs one retrieval, feeds the same 10 candidates to both pipelines, times each stage, and returns everything you need for the report.
Idea
Both pipelines share step (a): one bi-encoder search. Pipeline 1 takes its first 3 results; Pipeline 2 re-ranks all 10 and takes its first 3. Running the search once guarantees both pipelines start from exactly the same candidates. time.perf_counter() gives high-resolution timestamps; the difference between two calls is elapsed time.


In [34]:
import time
def run_pipelines(question, call_llm=True):
    """Run Pipeline 1 and Pipeline 2 on one question. call_llm=False skips Gemini (fast, free)."""
    t0 = time.perf_counter()
    candidates = baseline_retrieve(question)            # shared step: bi-encoder top 10
    t1 = time.perf_counter()
    reranked = rerank(question, candidates)             # Pipeline 2 only: cross-encoder on the 10
    t2 = time.perf_counter()
    result = {
        "question": question,
        "candidates": candidates,
        "reranked": reranked,
        "base_ids": [c["chunk_id"] for c in candidates[:TOP_N_CONTEXT]],   # Pipeline 1 context
        "rr_ids": [c["chunk_id"] for c in reranked[:TOP_N_CONTEXT]],       # Pipeline 2 context
        "retrieve_ms": (t1 - t0) * 1000,
        "rerank_ms": (t2 - t1) * 1000,
    }
    if call_llm:
        result["answer1"], result["llm1_ms"] = generate_answer(question, result["base_ids"])   # Answer 1
        result["answer2"], result["llm2_ms"] = generate_answer(question, result["rr_ids"])     # Answer 2
    return result

Cell 7.2 the rank-shift table

In [35]:
def rank_table(result):
    """One row per candidate: where the bi-encoder put it, where the cross-encoder put it."""
    rows = []
    for c in result["reranked"]:
        cid = c["chunk_id"]
        rows.append({
            "chunk_id": cid,
            "page": chunks[cid]["page"],
            "bi_rank": c["bi_rank"],
            "ce_rank": c["ce_rank"],
            "moved": c["bi_rank"] - c["ce_rank"],     # positive = promoted by the cross-encoder
            "bi_score": round(c["bi_score"], 3),
            "ce_score": round(c["ce_score"], 2),
            "preview": chunks[cid]["text"][:90] + "...",
        })
    return pd.DataFrame(rows).sort_values("bi_rank").reset_index(drop=True)

r = run_pipelines("Can I withdraw my consent?")
display(rank_table(r))
print("Pipeline 1 context:", r["base_ids"], "\nANSWER 1:\n", r["answer1"])
print("\nPipeline 2 context:", r["rr_ids"], "\nANSWER 2:\n", r["answer2"])

,chunk_id,page,bi_rank,ce_rank,moved,bi_score,ce_score,preview
0,222,41,1,6,-5,0.519,-3.15,with the following further information necessary to ensure fair and transparent processing...
1,206,37,2,1,1,0.490,4.10,"a manner which is clearly distinguishable from the other matters, in an intelligible and e..."
2,226,42,3,3,0,0.479,-1.74,"is based on point (f) of Article 6(1), the legitimate interests pursued by the controller ..."
3,41,8,4,2,2,0.469,1.70,not be regarded as freely given if the data subject has no genuine or free choice or is un...
4,40,8,5,4,1,0.392,-2.08,and the European Court of Human Rights. (42) Where processing is based on the data subject...
5,208,38,6,9,-3,0.357,-9.95,2. The controller shall make reasonable efforts to verify in such cases that consent is gi...
6,205,37,7,10,-3,0.352,-10.06,"which the personal data have been collected, in particular regarding the relationship betw..."
7,234,44,8,5,3,0.343,-3.07,(b) the data subject withdraws consent on which the processing is based according to point...
8,28,6,9,8,1,0.338,-8.29,Union or Member State law. The requests for disclosure sent by the public authorities shou...
9,68,12,10,7,3,0.331,-5.84,online services and online identifiers. A controller should not retain personal data for t...


Pipeline 1 context: [222, 206, 226] 
ANSWER 1:
 Yes, you have the right to withdraw your consent at any time [chunk 206, chunk 222, chunk 226].

**Conditions and Exceptions:**
*   This right applies where the processing is based on point (a) of Article 6(1) or point (a) of Article 9(2) [chunk 222, chunk 226].
*   The withdrawal of consent shall not affect the lawfulness of processing based on consent before its withdrawal [chunk 206, chunk 222, chunk 226].
*   It shall be as easy to withdraw as to give consent [chunk 206].

Pipeline 2 context: [206, 41, 226] 
ANSWER 2:
 Yes, the data subject shall have the right to withdraw his or her consent at any time [chunk 206, chunk 226].

**Conditions/Exceptions:**
* The withdrawal of consent shall not affect the lawfulness of processing based on consent before its withdrawal [chunk 206, chunk 226].
* Prior to giving consent, the data subject shall be informed of this right [chunk 206].
* It shall be as easy to withdraw as to give consent [chunk

How to read the table
• Rows are sorted by bi_rank, so the first three rows are Pipeline 1's context.
• Rows with ce_rank 1 to 3 are Pipeline 2's context.
• A row with bi_rank 4 to 10 and ce_rank 1 to 3 is a promotion: exactly what the assignment wants you to document, written as "chunk 41 moved from rank #8 to rank #1".
• A large negative moved on a top-3 chunk is a demotion: the bi-encoder liked it, the cross-encoder thinks it is off-topic. These are often your keyword traps.

Chapter 8: Hunting failure queries
The goal is at least two queries, ideally one per category below, where the gold chunk sits outside the baseline top 3 and the cross-encoder pulls it in. You find them by working backwards from the answer, not by guessing questions.
Idea
Start from a passage you already know is the answer (the gold chunk), then write a question a real user might ask about it. You can only call a result a failure if you know which chunk should have won, so the gold chunk is your ground truth. Expect to try 15 to 25 questions to get 4 to 6 good cases; that is normal and is the main time cost of this project.

The four categories and how to build each.
1. keyword trap : 
      what to look for in you PDF : 
            A topic whose answer uses different words from a section that repeats the user's words (e.g. the doc says "forfeit", another section says "penalty" many times)
      How to phrase the question : 
            Use the everyday word, which appears heavily in the wrong section

2. Exception or corner case: 
      what to look for in you PDF : 
            A general rule in one place and an "except when", "unless" or "only if" clause elsewhere
      How to phrase the question : 
            Put the special condition in the question: "Can I do X when Y?"

3. Negation or constraint: 
      what to look for in you PDF : 
        A list of things that are prohibited, or limits on a specific role

      How to phrase the question : 
           Ask what is not allowed, or what a specific role cannot do

4.   Buried at ranks 4 to  10: 
      what to look for in you PDF : 
            Any answer chunk the bi-encoder ranks low
      How to phrase the question : 
            Any natural phrasing: scan_queries flags these automatically.  


Cell 8.1 scan many questioons at once(no LLM calls)

In [ ]:
def scan_queries(questions):
    """ 
    For each quesiton :  both top-3 lists and any chunk promoted from rnak 4-10
    into the top 3.
    """

    rows = []
    for q in questions:
        r = run_pipelines(q, call_llm=False)
        promoted = [f"{c['chunk_id']} : #{c['bi_rank']} -> # {c['ce_rank']}" 
                    for c in r["reranked"][:TOP_N_CONTEXT] if c["bi_rank"] > TOP_N_CONTEXT]

        rows.append({"question":q, "baseline_top3": r["base_ids"], "reranked_top3":r["rr_ids"], 
                     "promoted_into_top3": promoted})
    return pd.DataFrame(rows)

# question -> gold chunk id (the chunk that actually contains the answer in the GDPR Articles)
gold = {
    "How quickly must a company report a hack?": 266,                                  # Art 33
    "Can a company process someone's health data?": 208,                               # Art 9
    "At what age can a child agree to an app using their data?": 207,                  # Art 8
    "When is a company not allowed to decide about me using only an algorithm?": 243,  # Art 22
    "When can a company refuse to delete my data?": 69,                               # Art 65
    "What is the maximum fine for breaking the GDPR?": 400,                            # Art 83(5)
    "Which organisations must appoint a data protection officer?": 280,                # Art 37
    "Can I take back my consent later?": 206,                                          # Art 7(3)
}
candidate_questions = list(gold)
scan_queries(candidate_questions)

,question,baseline_top3,reranked_top3,promoted_into_top3
0,How quickly must a company report a hack?,"[95, 94, 96]","[94, 267, 266]","[267 : #5 -> # 2, 266 : #6 -> # 3]"
1,Can a company process someone's health data?,"[66, 44, 169]","[44, 66, 101]",[101 : #6 -> # 3]
2,At what age can a child agree to an app using their data?,"[207, 62, 61]","[207, 36, 35]","[36 : #5 -> # 2, 35 : #9 -> # 3]"
3,When is a company not allowed to decide about me using only an algorithm?,"[76, 243, 78]","[76, 317, 79]","[317 : #5 -> # 2, 79 : #7 -> # 3]"
4,When can a company refuse to delete my data?,"[233, 234, 68]","[69, 233, 38]","[69 : #5 -> # 1, 38 : #8 -> # 3]"
5,What is the maximum fine for breaking the GDPR?,"[400, 399, 398]","[399, 400, 401]",[401 : #6 -> # 3]
6,Which organisations must appoint a data protection officer?,"[281, 280, 284]","[281, 280, 282]",[282 : #7 -> # 3]
7,Can I take back my consent later?,"[206, 41, 222]","[206, 41, 40]",[40 : #4 -> # 3]


Cell 8.2: check one question against its gold chunk

In [37]:
def gold_ranks(question, gold_id):
    """Where does the known-correct chunk land in each pipeline?"""
    r = run_pipelines(question, call_llm=False)
    bi = next((c["bi_rank"] for c in r["candidates"] if c["chunk_id"] == gold_id), None)
    ce = next((c["ce_rank"] for c in r["reranked"] if c["chunk_id"] == gold_id), None)
    if bi is None:
        print(f"Gold chunk {gold_id} is NOT in the baseline top {TOP_K_RETRIEVE}: the re-ranker never sees it.")
    else:
        print(f"Gold chunk {gold_id}: baseline rank #{bi} -> re-ranked #{ce}")
    return bi, ce

find_chunk("shall not apply to the extent")   # 1. locate the gold chunk by a phrase from the rule (Art 17(3))
# show_chunk(235)                             # 2. read it to confirm it answers the question
# gold_ranks("When can a company refuse to delete my data?", 235)   # 3. test the question

,chunk_id,page,preview
0,235,44,"paragraph 1 to erase the personal data, the controller, taking account of available technology and the cost of imple..."


In [38]:
# where does each gold chunk land in the baseline vs re-ranked list?
for q, g in gold.items():
    print(q)
    gold_ranks(q, g)
    print()

How quickly must a company report a hack?
Gold chunk 266: baseline rank #6 -> re-ranked #3

Can a company process someone's health data?
Gold chunk 208 is NOT in the baseline top 10: the re-ranker never sees it.

At what age can a child agree to an app using their data?
Gold chunk 207: baseline rank #1 -> re-ranked #1

When is a company not allowed to decide about me using only an algorithm?
Gold chunk 243: baseline rank #2 -> re-ranked #4

When can a company refuse to delete my data?
Gold chunk 69: baseline rank #5 -> re-ranked #1

What is the maximum fine for breaking the GDPR?
Gold chunk 400: baseline rank #1 -> re-ranked #2

Which organisations must appoint a data protection officer?
Gold chunk 280: baseline rank #2 -> re-ranked #2

Can I take back my consent later?
Gold chunk 206: baseline rank #1 -> re-ranked #1



Cell 8.3: record a full case study (calls the LLM twice)

In [ ]:
case_log = []   # every case study is saved here for the report

def case_study(category, question, gold_id=None):
    """Rank table + Answer 1 vs Answer 2 for one failure query, logged to case_log."""
    r = run_pipelines(question)
    print(f"[{category}] {question}\n")
    display(rank_table(r))
    if gold_id is not None:
        gold_ranks(question, gold_id)
    print("\nPipeline 1 context:", r["base_ids"], "\nANSWER 1:\n", r["answer1"])
    print("\nPipeline 2 context:", r["rr_ids"], "\nANSWER 2:\n", r["answer2"])
    case_log.append({"category": category, "question": question, "gold_id": gold_id,
                     "baseline_top3": r["base_ids"], "reranked_top3": r["rr_ids"],
                     "answer1": r["answer1"], "answer2": r["answer2"],
                     "llm1_ms": r["llm1_ms"], "llm2_ms": r["llm2_ms"]})
    return r

# One cell per case, so each output stays visible in the notebook, e.g.
# case_study("Negation", "When can a company refuse to delete my data?", gold_id=69)

In [40]:
find_chunk("72 hours")   # note the chunk_id whose text is Article 33 (breach notification)

,chunk_id,page,preview
0,94,17,"aware that a personal data breach has occurred, the controller should notify the personal data breach to the supervi..."
1,266,52,by which to demonstrate compliance with the requirements set out in paragraph 1 of this Article. 4. The controller a...
2,267,52,the rights and freedoms of natural persons. Where the notification to the supervisory authority is not made within 7...


In [41]:
case_study("Keyword trap / recital distractor", "How quickly must a company report a hack?", gold_id=266);

[Keyword trap / recital distractor] How quickly must a company report a hack?



,chunk_id,page,bi_rank,ce_rank,moved,bi_score,ce_score,preview
0,95,17,1,4,-3,0.449,-10.02,allow him or her to take the necessary precautions. The communication should describe the ...
1,94,17,2,1,1,0.418,-6.33,"aware that a personal data breach has occurred, the controller should notify the personal ..."
2,96,17,3,7,-4,0.407,-10.68,supervisory authority and the data subject. The fact that the notification was made withou...
3,268,52,4,5,-1,0.405,-10.11,"breach, including, where appropriate, measures to mitigate its possible adverse effects. 4..."
4,267,52,5,2,3,0.364,-8.26,the rights and freedoms of natural persons. Where the notification to the supervisory auth...
5,266,52,6,3,3,0.362,-9.25,by which to demonstrate compliance with the requirements set out in paragraph 1 of this Ar...
6,339,68,7,10,-3,0.346,-11.16,"the extent appropriate, the subject matter of the complaint and inform the complainant of ..."
7,377,77,8,8,0,0.341,-10.99,criteria and conditions for decisions based on profiling pursuant to Article 22(2); (g) is...
8,336,67,9,6,3,0.335,-10.12,"to handle a complaint lodged with it or a possible infringement of this Regulation, if the..."
9,149,26,10,9,1,0.330,-11.13,"to lodge a complaint in that Member State, independently of a data subject's mandate, and ..."


Gold chunk 266: baseline rank #6 -> re-ranked #3

Pipeline 1 context: [95, 94, 96] 
ANSWER 1:
 A company (controller) must report a personal data breach to the supervisory authority without undue delay and, where feasible, not later than 72 hours after becoming aware of it [chunk 94].

**Exceptions and Conditions:**
*   **No notification to supervisory authority:** Notification is not required if the controller can demonstrate that the personal data breach is unlikely to result in a risk to the rights and freedoms of natural persons [chunk 94].
*   **Delay in notification to supervisory authority:** If notification cannot be achieved within 72 hours, the reasons for the delay should accompany the notification, and information may be provided in phases without undue further delay [chunk 94].
*   **Communication to data subjects:** The controller should communicate a personal data breach to the data subject without undue delay, where that breach is likely to result in a high risk to the 

In [42]:
case_study("Negation", "When can a company refuse to delete my data?", gold_id=69);

[Negation] When can a company refuse to delete my data?



,chunk_id,page,bi_rank,ce_rank,moved,bi_score,ce_score,preview
0,233,43,1,2,-1,0.634,-3.84,The data subject shall have the right to obtain from the controller the erasure of persona...
1,234,44,2,5,-3,0.564,-7.03,(b) the data subject withdraws consent on which the processing is based according to point...
2,68,12,3,4,-1,0.547,-6.05,online services and online identifiers. A controller should not retain personal data for t...
3,232,43,4,8,-4,0.545,-8.46,"the data subject, the controller may charge a reasonable fee based on administrative costs..."
4,69,13,5,1,4,0.543,-3.01,his or her consent as a child and is not fully aware of the risks involved by the processi...
5,70,13,6,6,0,0.539,-7.14,defence of legal claims. (66) To strengthen the right to be forgotten in the online enviro...
6,199,36,7,10,-3,0.517,-9.96,(e) kept in a form which permits identification of data subjects for no longer than is nec...
7,38,7,8,3,5,0.514,-5.69,be processed only if the purpose of the processing could not reasonably be fulfilled by ot...
8,235,44,9,7,2,0.514,-7.54,"paragraph 1 to erase the personal data, the controller, taking account of available techno..."
9,230,43,10,9,1,0.505,-9.36,Article 15 Right of access by the data subject 1. The data subject shall have the right to...


Gold chunk 69: baseline rank #5 -> re-ranked #1

Pipeline 1 context: [233, 234, 68] 
ANSWER 1:
 A company can refuse to delete your data if:
* The data subject withdraws consent on which the processing is based, but there is another legal ground for the processing [chunk 234].
* The data subject objects to the processing pursuant to Article 21(1), but there are overriding legitimate grounds for the processing [chunk 234].

Pipeline 2 context: [69, 233, 38] 
ANSWER 2:
 A company (controller) can lawfully retain personal data, and thus refuse to delete it, where it is necessary for:
*   Exercising the right of freedom of expression and information [chunk 69].
*   Compliance with a legal obligation [chunk 69].
*   The performance of a task carried out in the public interest or in the exercise of official authority vested in the controller [chunk 69].
*   Grounds of public interest in the area of public health [chunk 69].
*   Archiving purposes in the public interest, scientific or histori

In [43]:
case_study("Counter-example (re-ranking hurt)", "When is a company not allowed to decide about me using only an algorithm?", gold_id=243);

[Counter-example (re-ranking hurt)] When is a company not allowed to decide about me using only an algorithm?



,chunk_id,page,bi_rank,ce_rank,moved,bi_score,ce_score,preview
0,76,14,1,1,0,0.460,-6.85,"(71) The data subject should have the right not to be subject to a decision, which may inc..."
1,243,46,2,4,-2,0.421,-8.85,"public interest. Article 22 Automated individual decision-making, including profiling 1. T..."
2,78,14,3,6,-3,0.393,-9.56,measure should not concern a child. In order to ensure fair and transparent processing in ...
3,240,45,4,8,-4,0.384,-10.71,task carried out in the public interest or in the exercise of official authority vested in...
4,317,63,5,2,3,0.372,-8.63,"by default, legal basis for processing, processing of special categories of personal data,..."
5,226,42,6,9,-3,0.371,-11.06,"is based on point (f) of Article 6(1), the legitimate interests pursued by the controller ..."
6,79,14,7,3,4,0.364,-8.76,Automated decision-making and profiling based on special categories of personal data shoul...
7,44,8,8,10,-2,0.354,-11.24,"in the public interest to do so, including for health purposes such as public health and s..."
8,242,46,9,7,2,0.348,-9.82,"4. At the latest at the time of the first communication with the data subject, the right r..."
9,41,8,10,5,5,0.347,-9.26,not be regarded as freely given if the data subject has no genuine or free choice or is un...


Gold chunk 243: baseline rank #2 -> re-ranked #4

Pipeline 1 context: [76, 243, 78] 
ANSWER 1:
 A company is generally not allowed to decide about you using only an algorithm if the decision is based solely on automated processing, including profiling, and produces legal effects concerning you or similarly significantly affects you [chunk 76, chunk 243].

However, this right not to be subject to such a decision does not apply if the decision:
* Is necessary for entering into, or performance of, a contract between the data subject and a data controller [chunk 243].
* Is authorised by Union or Member State law to which the controller is subject and which also lays down suitable measures to safeguard the data subject's rights and freedoms and legitimate interests (e.g., for fraud and tax-evasion monitoring and prevention purposes) [chunk 76, chunk 243].
* Is based on the data subject's explicit consent [chunk 243].

**Exceptions/Conditions:**
* The measure should not concern a child [chun

In [44]:
case_study("Recall limit (gold not in top 10)", "Can a company process someone's health data?", gold_id=208);

[Recall limit (gold not in top 10)] Can a company process someone's health data?



,chunk_id,page,bi_rank,ce_rank,moved,bi_score,ce_score,preview
0,66,12,1,2,-1,0.577,-0.36,"access to personal data which have been collected concerning him or her, and to exercise t..."
1,44,8,2,1,1,0.521,1.35,"in the public interest to do so, including for health purposes such as public health and s..."
2,169,29,3,10,-7,0.511,-6.79,"to restriction of processing, to data portability, and to object when processing personal ..."
3,30,6,4,9,-5,0.500,-6.40,Data subjects should have the opportunity to give their consent only to certain areas of r...
4,191,34,5,8,-3,0.497,-5.52,"leading to the accidental or unlawful destruction, loss, alteration, unauthorised disclosu..."
5,101,18,6,3,3,0.495,-1.29,scale. The processing of personal data should not be considered to be on a large scale if ...
6,32,6,7,7,0,0.494,-5.14,on the purposes and means of the processing of personal data are taken in another establis...
7,57,10,8,4,4,0.487,-3.23,"the area of public health. Therefore, this Regulation should provide for harmonised condit..."
8,43,8,9,5,4,0.470,-3.84,in the exercise of an official authority may be sufficient. It should also be for Union or...
9,241,45,10,6,4,0.464,-4.55,"personal data are processed for direct marketing purposes, the data subject shall have the..."


Gold chunk 208 is NOT in the baseline top 10: the re-ranker never sees it.

Pipeline 1 context: [66, 44, 169] 
ANSWER 1:
 Yes, a company can process someone's health data under certain conditions.

Processing of personal data, which includes health data, is regarded as lawful where it is:
*   In the public interest to do so, including for health purposes such as public health and social protection and the management of health care services [chunk 44].
*   Necessary to protect an interest which is essential for the life of the data subject or that of another natural person [chunk 44].
*   For archiving purposes in the public interest, scientific or historical research purposes or statistical purposes. This processing must comply with specific conditions and safeguards, and may entail specific procedures for data subjects to exercise their rights, along with technical and organisational measures to minimise processing. For scientific purposes, it should also comply with other relevant le

Cell 8.4: summary of all cases

In [45]:
cases_df = pd.DataFrame(case_log)
cases_df.to_csv("failure_cases.csv", index=False)   # keep this file for writing the report
cases_df[["category", "question", "gold_id", "baseline_top3", "reranked_top3"]]

,category,question,gold_id,baseline_top3,reranked_top3
0,Keyword trap / recital distractor,How quickly must a company report a hack?,266,"[95, 94, 96]","[94, 267, 266]"
1,Negation,When can a company refuse to delete my data?,69,"[233, 234, 68]","[69, 233, 38]"
2,Counter-example (re-ranking hurt),When is a company not allowed to decide about me using only an algorithm?,243,"[76, 243, 78]","[76, 317, 79]"
3,Recall limit (gold not in top 10),Can a company process someone's health data?,208,"[66, 44, 169]","[44, 66, 101]"


The workflow, step by step
1. Skim your PDF and note 10 tricky facts: exceptions, prohibitions, penalties, role limits.
2. For each, run find_chunks with a distinctive word and confirm the gold chunk with show_chunk.
3. Write one question per fact, phrased as a user who has not read the document would ask it.
4. Run them through scan_queries, then gold_ranks for the promising ones.
5. Gold chunk already in the baseline top 3: rephrase with everyday synonyms instead of the document's own terms.
6. Gold chunk not in the top 10 at all: rephrase closer to the document, or try CHUNK_WORDS = 120 (re-run Chapters 3 to 4). Write down that this happened; it shows the recall limit from Chapter 1.
7. When the gold chunk moves from rank 4 to 10 into the top 3, run case_study on it in its own cell.
8. Read Answer 1 against the gold chunk: is a condition missing, a rule misstated, a fact invented, or "Not found"?
Record honestly. If the cross-encoder pushes a gold chunk down, keep that case too: a counter-example shows you understood the method, not just the expected result.

Chapter 9: Latency profiling and the scaling experiment
You measure three things: retrieval time, re-ranking time, and LLM time, then extrapolate what cross-encoding the whole collection would cost.
Idea
• Warm-up: the first call to any model is slow (memory allocation, lazy loading). We run one throwaway query first, so it does not distort the numbers.
• Repeat and use the median: single timings jump around. Median is the typical case; p95 (the 95th percentile) shows the slow tail.
• Report the hardware: CPU versus T4 GPU can change cross-encoder time by 5 to 10 times.
• LLM time is measured separately, because network delay to Gemini usually dwarfs both retrieval stages and would hide their difference.

Cell 9.1: retrieval vs re-ranking latency

In [46]:
def profile_latency(questions, repeats=5):
    """Time the retrieval stages over many runs (no LLM). Returns median, p95 and mean in ms."""
    run_pipelines(questions[0], call_llm=False)    # warm-up run, not recorded
    rows = []
    for q in questions:
        for _ in range(repeats):
            r = run_pipelines(q, call_llm=False)
            rows.append({"retrieve_ms": r["retrieve_ms"], "rerank_ms": r["rerank_ms"]})
    df = pd.DataFrame(rows)
    df["p1_total_ms"] = df["retrieve_ms"]                     # Pipeline 1 = retrieval only
    df["p2_total_ms"] = df["retrieve_ms"] + df["rerank_ms"]   # Pipeline 2 = retrieval + re-rank
    summary = pd.DataFrame({"median_ms": df.median(), "p95_ms": df.quantile(0.95),
                            "mean_ms": df.mean()}).round(1)
    summary.index = ["Bi-encoder retrieval (embed query + FAISS top-10)",
                     "Cross-encoder re-rank (10 pairs)",
                     "Pipeline 1 retrieval total",
                     "Pipeline 2 retrieval total (retrieve + re-rank)"]
    return summary

latency_table = profile_latency(candidate_questions, repeats=5)
display(latency_table)

,median_ms,p95_ms,mean_ms
Bi-encoder retrieval (embed query + FAISS top-10),12.2,13.6,12.0
Cross-encoder re-rank (10 pairs),307.0,331.3,308.5
Pipeline 1 retrieval total,12.2,13.6,12.0
Pipeline 2 retrieval total (retrieve + re-rank),318.8,344.1,320.6


Cell 9.2: LLM time from your case studies

In [ ]:
# Median generation time across the case studies (only successful calls are timed)
print("LLM time, Answer 1 (ms):", round(cases_df["llm1_ms"].median()))
print("LLM time, Answer 2 (ms):", round(cases_df["llm2_ms"].median()))

LLM time, Answer 1 (ms): 5234
LLM time, Answer 2 (ms): 5207


Cell 9.3: what if we cross-encoded everything?

In [48]:
def cross_encode_everything(question):
    """Time the cross-encoder on EVERY chunk, then extrapolate to bigger collections."""
    pairs = [(question, c["text"]) for c in chunks]
    t0 = time.perf_counter()
    cross_encoder.predict(pairs)
    total_ms = (time.perf_counter() - t0) * 1000
    per_pair_ms = total_ms / len(pairs)
    print(f"Cross-encoding ALL {len(pairs)} chunks took {total_ms:.0f} ms ({per_pair_ms:.2f} ms per pair)")
    for n in [10_000, 1_000_000, 100_000_000]:
        secs = per_pair_ms * n / 1000
        print(f"  extrapolated for {n:>11,} chunks: {secs:,.0f} s (~{secs/3600:,.1f} h) PER QUERY")

cross_encode_everything(candidate_questions[0])

Cross-encoding ALL 419 chunks took 12502 ms (29.84 ms per pair)
  extrapolated for      10,000 chunks: 298 s (~0.1 h) PER QUERY
  extrapolated for   1,000,000 chunks: 29,838 s (~8.3 h) PER QUERY
  extrapolated for 100,000,000 chunks: 2,983,755 s (~828.8 h) PER QUERY
